# Teacher labeling with local MedGemma-27B (Kaggle/Colab, GPU)

Local-inference equivalent of `src/teacher_labeling/generate_aux_labels.py`, for
running MedGemma as the primary teacher without standing up a vLLM server.

It reuses the exact same prompt, JSON schema, validation rules, and
majority-vote logic as `generate_aux_labels.py`, and writes output in the
**same JSONL record shape** (`id`, `gold_label`, `split`, `text`, `model`,
`base_url`, `runs`, `majority_label`, `majority_meta`,
`requires_human_audit`), so it plugs straight into:

```
this notebook
        |
        v
data/synthetic_aux/*_raw_primary_runs.jsonl
        |
        v
src/teacher_judging/judge_aux_labels.py
        |
        v
src/teacher_labeling/apply_quality_gates.py
        |
        v
src/teacher_labeling/build_student_sft_jsonl.py
```

### Inputs expected (attach as a Kaggle dataset, or upload to `/kaggle/input`)
Either one works — the prep cell (1b) auto-detects which you gave it:
- **raw** CLPsych "Moments of Change" timeline JSON — a single array file like
  `compiled_01.json`, or a folder of per-timeline JSON files. Cell 1b converts
  it to a split CSV for you.
- **or** an already-split CSV from `create_data_splits.py`, e.g.
  `data/processed/clpsych22/splits/all_with_splits.csv` (columns: `id`,
  `gold_label`, `text`, `split`, ...).

### Required setup
- Accelerator: GPU with >=24GB VRAM (a single T4/P100 16GB is **not** enough for `google/medgemma-27b-text-it` even in 4-bit, once weights, KV cache for `MAX_NEW_TOKENS=1024` x `RUNS=3`, and activation memory are counted). Use Kaggle's dual-T4 (2x16GB, `device_map="auto"` will shard across both) or a Colab/Kaggle A100 (40GB).
- Secret `HF_TOKEN` — MedGemma-27B is gated behind the Health AI Developer Foundation terms of use, you must accept them on the
  `google/medgemma-27b-text-it` model card first, then add the token as a Kaggle secret.

In [ ]:
# === CELL 1 — install ===
import sys, subprocess, torch
cap = torch.cuda.get_device_capability(0)
print("GPU capability:", cap)
assert cap >= (7, 0), "Use a GPU with bf16 support (e.g. T4)."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers", "accelerate", "bitsandbytes", "huggingface_hub"], check=False)
print("install done")

## 1b. Build the split CSV from raw CLPsych "Moments of Change" timeline JSON

This ports `src/teacher_labeling/prepare_clpsych22_timelines.py` and
`create_data_splits.py` inline, so you can attach the **raw** timeline data
(e.g. `compiled_01.json`, an array of timelines, or a folder of per-timeline
JSON files) as the Kaggle dataset and skip the local prep entirely.

It writes `/kaggle/working/clpsych22_splits/all_with_splits.csv`, which the
next cell picks up automatically. `Switch`/`Escalation`/`Well-being` are
carried into `gold_label` as **context only** (not suicide-risk tiers); the
teacher re-derives `risk_tier` from the post text per its own rubric.

If you instead attached an already-split `all_with_splits.csv`, this cell
finds no raw timelines and harmlessly does nothing — the next cell uses your
CSV directly.

In [ ]:
# === CELL 1b — raw CLPsych timeline JSON -> posts CSV -> train/dev/test split ===
# Ports prepare_clpsych22_timelines.py + create_data_splits.py (no repo clone needed).
import os, glob, json, csv, random
from collections import defaultdict

# ---- prep config (matches the repo defaults) ----
CONTEXT_WINDOW = 2        # preceding posts from same timeline to prepend as context
MAX_CHARS      = 12000    # truncate middle of over-long rows
TRAIN_RATIO    = 0.70
DEV_RATIO      = 0.15
SEED           = 42

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
PREP_DIR = f"{OUT_DIR}/clpsych22_splits"
PREPARED_CSV = None  # set below if raw timelines are found

def _load_timelines():
    """Find raw timeline JSON under /kaggle/input. Accepts either a single file
    that is an array of timelines (e.g. compiled_01.json) or many per-timeline
    files, each a dict with a 'posts' list."""
    candidates = sorted(glob.glob("/kaggle/input/**/*.json", recursive=True)) \
        or sorted(glob.glob("./*.json"))
    timelines = []
    for path in candidates:
        try:
            data = json.load(open(path, encoding="utf-8"))
        except Exception:
            continue
        items = data if isinstance(data, list) else [data]
        for tl in items:
            if isinstance(tl, dict) and isinstance(tl.get("posts"), list):
                timelines.append((tl.get("timeline_id") or os.path.splitext(os.path.basename(path))[0], tl))
    return timelines

def _format_gold_label(post):
    switch = post.get("Switch") or "0"
    escalation = post.get("Escalation") or "0"
    wb = post.get("Well-being")
    wb_str = "n/a" if wb is None else str(wb)
    return ("CLPsych22 Moments-of-Change annotation (not a suicide-risk tier): "
            f"Switch={switch}, Escalation={escalation}, Well-being={wb_str}")

def _build_context_text(posts, target_index, window):
    start = max(0, target_index - window) if window > 0 else target_index
    frags, included = [], 0
    for post in posts[start:target_index]:
        t = (post.get("post") or "").strip()
        if not t:
            continue
        frags.append(f"[earlier post, {post.get('date', 'unknown date')}] {t}")
        included += 1
    target = posts[target_index]
    frags.append(f"[target post, {target.get('date', 'unknown date')}] {(target.get('post') or '').strip()}")
    return "\n\n".join(frags), included

def _truncate(text, max_chars):
    if max_chars <= 0 or len(text) <= max_chars:
        return text, False
    half = max_chars // 2
    head, tail = text[:half].rstrip(), text[-half:].lstrip()
    marker = "\n\n[... middle omitted for length; beginning and end retained ...]\n\n"
    return head + marker + tail, True

def _ratio_to_counts(n, train_ratio, dev_ratio):
    train_n = round(n * train_ratio)
    dev_n = round(n * dev_ratio)
    if train_n + dev_n > n:
        dev_n = max(0, n - train_n)
    return train_n, dev_n, n - train_n - dev_n

timelines = _load_timelines()
if not timelines:
    print("No raw CLPsych timeline JSON found under /kaggle/input — "
          "assuming you attached an already-split CSV; the next cell will locate it.")
else:
    # --- 1) timelines -> one row per post ---
    fieldnames = ["id", "gold_label", "source", "text", "timeline_id", "post_id",
                  "post_index", "date", "switch", "escalation", "well_being",
                  "context_posts_included", "chars_before_truncation", "truncated"]
    rows = []
    for timeline_id, tl in timelines:
        posts = tl.get("posts") or []
        for index, post in enumerate(posts):
            post_text = (post.get("post") or "").strip()
            if not post_text:
                continue
            text, ctx = _build_context_text(posts, index, CONTEXT_WINDOW)
            orig_len = len(text)
            text, truncated = _truncate(text, MAX_CHARS)
            rows.append({
                "id": f"{timeline_id}__{post.get('post_id', index)}",
                "gold_label": _format_gold_label(post),
                "source": "clpsych22-moments-of-change",
                "text": text,
                "timeline_id": timeline_id,
                "post_id": post.get("post_id", ""),
                "post_index": post.get("post_index", index),
                "date": post.get("date", ""),
                "switch": post.get("Switch") or "0",
                "escalation": post.get("Escalation") or "0",
                "well_being": post.get("Well-being"),
                "context_posts_included": ctx,
                "chars_before_truncation": orig_len,
                "truncated": str(truncated),
            })
    print(f"timelines={len(timelines)} posts(rows)={len(rows)}")

    # --- 2) train/dev/test split, grouped by timeline_id (no leakage) ---
    grouped = defaultdict(list)
    for r in rows:
        grouped[r["timeline_id"]].append(r)
    label_to_groups = defaultdict(list)
    for gid, grows in grouped.items():
        labels = [r["gold_label"] for r in grows]
        label = max(set(labels), key=labels.count) if labels else ""
        label_to_groups[label].append(gid)

    rng = random.Random(SEED)
    split_for_group = {}
    for label, gids in sorted(label_to_groups.items()):
        ids = list(gids); rng.shuffle(ids)
        tr, dv, _ = _ratio_to_counts(len(ids), TRAIN_RATIO, DEV_RATIO)
        for gid in ids[:tr]:            split_for_group[gid] = "train"
        for gid in ids[tr:tr+dv]:       split_for_group[gid] = "dev"
        for gid in ids[tr+dv:]:         split_for_group[gid] = "test"

    split_fieldnames = fieldnames + ["split"]
    counts = defaultdict(int)
    os.makedirs(PREP_DIR, exist_ok=True)
    PREPARED_CSV = f"{PREP_DIR}/all_with_splits.csv"
    with open(PREPARED_CSV, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=split_fieldnames)
        w.writeheader()
        for gid, grows in grouped.items():
            sp = split_for_group[gid]
            for r in grows:
                out = dict(r); out["split"] = sp
                w.writerow(out); counts[sp] += 1
    print("split counts:", dict(counts), "->", PREPARED_CSV)


In [ ]:
# === CELL 2 — HF auth + locate input CSV ===
import os, glob
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("HF login OK (secret)")
except Exception:
    import getpass
    login(getpass.getpass("HF token (MedGemma-27B is gated, required): "))

# --- EDIT if your filename/location differs ---
# Prefer the CSV built by the prep cell above; otherwise locate an attached one.
if "PREPARED_CSV" in dir() and PREPARED_CSV:
    INPUT_CSV = PREPARED_CSV
else:
    cands = sorted(glob.glob("/kaggle/input/**/all_with_splits.csv", recursive=True)) \
        or sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)) \
        or sorted(glob.glob("./all_with_splits.csv"))
    assert cands, "Attach raw CLPsych timeline JSON (handled by the prep cell) or a split CSV."
    INPUT_CSV = cands[0]
print("using input csv:", INPUT_CSV)

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUTPUT_JSONL = f"{OUT_DIR}/raw_primary_runs_medgemma27b.jsonl"

ID_COLUMN = "id"
TEXT_COLUMN = "text"
GOLD_COLUMN = "gold_label"
SPLIT_COLUMN = "split"
ALLOWED_SPLITS = {"train", "dev"}
RUNS = 3
TEMPERATURE = 0.4
MAX_NEW_TOKENS = 1024
LIMIT = 20  # smoke test first; set to 0 for the full split

## 3. Load MedGemma-27B

Same `BitsAndBytesConfig` shape used elsewhere in this project's notebooks: `load_in_4bit=True, bnb_4bit_quant_type="nf4"`. This is bitsandbytes' NF4 4-bit scheme, the `transformers`-native equivalent of a GGUF Q4_K_M quant — similar bit-width/quality tradeoff, no llama.cpp dependency needed. If VRAM is still too tight on your accelerator, a GGUF Q4_K_M build (e.g. `bartowski/google_medgemma-27b-it-GGUF`) run through `llama-cpp-python` is the fallback, but isn't wired up here.

In [ ]:
# === CELL 3 — load model ===
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

TEACHER_ID = "google/medgemma-27b-text-it"  # text-only instruction-tuned 27B variant

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)

tok = AutoTokenizer.from_pretrained(TEACHER_ID)
model = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID, quantization_config=bnb, device_map="auto",
    torch_dtype=torch.bfloat16, attn_implementation="eager")
model.eval()
print("MedGemma-27B loaded")

## 4. Prompt, JSON extraction, validation, majority vote

Ported directly from `src/teacher_labeling/generate_aux_labels.py` so the
output is identical in shape and validation strictness, just generated
locally instead of over `/chat/completions`.

In [ ]:
# === CELL 4 — ported logic from generate_aux_labels.py ===
import json

SYSTEM_PROMPT = """You are an expert clinical NLP annotator creating structured rationale labels for a research-only crisis triage dataset.

Return only valid JSON matching the requested schema.

This is not a diagnosis, not therapy, and not emergency-service dispatch. Use observed evidence only.

Risk rubric:
- 0 No Acute Crisis: distress or support-seeking without self-harm intent, plan, imminent danger, or recent attempt.
- 1 Low Risk: passive ideation or vague hopelessness; no method, intent, plan, preparation, or recent behavior.
- 2 Moderate Risk: active ideation or method mentioned; no clear immediate intent or preparation.
- 3 High / Imminent Risk: intent plus plan/preparation, recent attempt, access to means with time-bound threat, or disclosed current attempt.

Dataset-label guidance:
- If the original label is from C-SSRS, use it as helpful but non-binding context.
- If the original label is from eRisk26 Task 2, label 0 means control and label 1 means depression-risk positive. This is not a suicide-risk tier. Do not assume that depression-positive means suicidal or high/imminent risk.
- If the original label mentions CLPsych22 Moments-of-Change (Switch/Escalation/Well-being), treat it as a mood/outlook-change annotation, not a suicide-risk tier.
- Always assign the final risk_tier from the text evidence and the rubric above.

Rules:
- evidence_spans must be exact substrings copied from the input text.
- risk_factors must be observed textual cues only. Do not infer hidden intent.
- protective_factors must be observed buffers only, such as support network, future plans, help-seeking, or stated reasons for living.
- clinical_rationale must be concise, 2-4 sentences.
- plain_language_summary must be understandable to a non-clinician.
- Do not provide diagnosis, medication advice, therapy instructions, or moral judgment.
- If uncertain, add uncertainty_flags instead of inventing facts.
- recommended_next_step must be one of: Supportive, Professional support, Urgent support, Emergency escalation, Human review."""

REQUIRED_KEYS = [
    "risk_tier", "confidence", "evidence_spans", "risk_factors", "protective_factors",
    "cssrs_axes", "clinical_rationale", "plain_language_summary",
    "recommended_next_step", "escalation_required", "uncertainty_flags",
]

# MedGemma-27B follows instructions more reliably than the 4B variant, but
# the explicit key skeleton is kept as a cheap safeguard against the same
# silent-key-dropping failure mode seen on smaller local teachers.
JSON_SKELETON_REMINDER = """You MUST output a single JSON object with EXACTLY these 11 top-level keys. Do not omit any of them, even if a value is an empty array or a default guess:
{
  "risk_tier": <integer 0-3>,
  "confidence": <number 0-1>,
  "evidence_spans": [<exact substrings from the input text, at least 1>],
  "risk_factors": [<strings, can be empty>],
  "protective_factors": [<strings, can be empty>],
  "cssrs_axes": {"ideation": "<string>", "behavior": "<string>", "intensity": "<string>", "lethality": "<string>", "precipitants": "<string>"},
  "clinical_rationale": "<2-4 sentences>",
  "plain_language_summary": "<string>",
  "recommended_next_step": "<one of: Supportive, Professional support, Urgent support, Emergency escalation, Human review>",
  "escalation_required": <true or false>,
  "uncertainty_flags": [<strings, can be empty>]
}
Before answering, check your draft JSON has all 11 keys listed above. If any is missing, add it."""

def build_user_prompt(text, gold_label):
    gold = gold_label or "unknown"
    return (
        "Create a structured crisis-triage rationale JSON for the input below.\n\n"
        f"Original dataset label, if any: {gold}\n"
        "Treat the original label as context, not as guaranteed truth.\n\n"
        "Input text:\n"
        "<TEXT>\n"
        f"{text}\n"
        "</TEXT>\n\n"
        f"{JSON_SKELETON_REMINDER}"
    )

def extract_json(content):
    content = content.strip()
    if content.startswith("```"):
        content = content.strip("`")
        if content.lower().startswith("json"):
            content = content[4:].strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        start = content.find("{")
        end = content.rfind("}")
        if start >= 0 and end > start:
            return json.loads(content[start : end + 1])
        raise

def exact_span_in_text(span, text):
    return bool(span) and span in text

def validate_aux_label(label, text):
    errors = []
    for key in REQUIRED_KEYS:
        if key not in label:
            errors.append(f"missing:{key}")
    if errors:
        return errors
    if label["risk_tier"] not in [0, 1, 2, 3]:
        errors.append("bad:risk_tier")
    if not isinstance(label["confidence"], (int, float)) or not 0 <= label["confidence"] <= 1:
        errors.append("bad:confidence")
    if not isinstance(label["evidence_spans"], list) or not label["evidence_spans"]:
        errors.append("bad:evidence_spans_empty")
    else:
        for span in label["evidence_spans"]:
            if not isinstance(span, str) or not exact_span_in_text(span, text):
                errors.append(f"bad:evidence_not_exact:{span[:60]}")
    if not isinstance(label["escalation_required"], bool):
        errors.append("bad:escalation_required")
    if label["recommended_next_step"] not in [
        "Supportive", "Professional support", "Urgent support", "Emergency escalation", "Human review",
    ]:
        errors.append("bad:recommended_next_step")
    return errors

def choose_majority(runs):
    valid = [r for r in runs if not r.get("validation_errors") and isinstance(r.get("label"), dict)]
    if not valid:
        return None, {"valid_runs": 0, "majority_tier": None, "agreement": 0.0}
    tiers = [r["label"]["risk_tier"] for r in valid]
    majority_tier = max(set(tiers), key=tiers.count)
    candidates = [r for r in valid if r["label"]["risk_tier"] == majority_tier]
    candidates.sort(key=lambda r: r["label"].get("confidence", 0), reverse=True)
    agreement = tiers.count(majority_tier) / len(tiers)
    meta = {
        "valid_runs": len(valid),
        "majority_tier": majority_tier,
        "agreement": agreement,
        "tier_votes": {str(t): tiers.count(t) for t in sorted(set(tiers))},
    }
    return candidates[0]["label"], meta

def already_done_ids(path):
    done = set()
    if not os.path.exists(path):
        return done
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                done.add(json.loads(line)["id"])
            except (json.JSONDecodeError, KeyError):
                continue
    return done

print("logic loaded")

## 5. Local generation loop

Same `--runs` x majority-vote structure as `generate_aux_labels.py`, but
calling `model.generate()` directly instead of an HTTP `/chat/completions`
request. Resumable: re-running this cell skips ids already in
`OUTPUT_JSONL`.

In [ ]:
# === CELL 5 — generation loop ===
import csv

@torch.inference_mode()
def generate_one(text, gold_label, temperature):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(text, gold_label)},
    ]
    enc = tok.apply_chat_template(messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
        do_sample=temperature > 0, temperature=max(temperature, 1e-5),
        pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

with open(INPUT_CSV, "r", encoding="utf-8-sig", newline="") as f:
    rows = [r for r in csv.DictReader(f) if r.get(SPLIT_COLUMN, "") in ALLOWED_SPLITS]
if LIMIT:
    rows = rows[:LIMIT]
print(f"rows to process: {len(rows)}")

done = already_done_ids(OUTPUT_JSONL)
with open(OUTPUT_JSONL, "a", encoding="utf-8") as out:
    for index, row in enumerate(rows, start=1):
        item_id = row.get(ID_COLUMN) or f"row-{index}"
        if item_id in done:
            continue
        text = row.get(TEXT_COLUMN, "")
        gold_label = row.get(GOLD_COLUMN, "")
        runs = []
        for run_index in range(RUNS):
            run_record = {"run_index": run_index}
            try:
                content = generate_one(text, gold_label, TEMPERATURE)
                label = extract_json(content)
                errors = validate_aux_label(label, text)
                run_record.update({"raw_content": content, "label": label, "validation_errors": errors})
            except Exception as exc:
                run_record.update({"error": str(exc), "validation_errors": ["exception"]})
            runs.append(run_record)

        majority, majority_meta = choose_majority(runs)
        record = {
            "id": item_id,
            "gold_label": gold_label,
            "split": row.get(SPLIT_COLUMN, ""),
            "text": text,
            "model": TEACHER_ID,
            "base_url": "local-kaggle-medgemma27b",
            "runs": runs,
            "majority_label": majority,
            "majority_meta": majority_meta,
            "requires_human_audit": bool(
                majority
                and (
                    majority.get("risk_tier") == 3
                    or majority.get("escalation_required")
                    or majority_meta.get("agreement", 0) < 1.0
                )
            ),
        }
        out.write(json.dumps(record, ensure_ascii=False) + "\n")
        out.flush()
        print(f"[{index}/{len(rows)}] wrote {item_id} tier={majority_meta.get('majority_tier')}")

print("done ->", OUTPUT_JSONL)

## 6. Quick quality check before scaling up

Check the invalid-JSON rate and tier distribution on this smoke-test batch
before setting `LIMIT = 0` and rerunning cell 5 on the full split.

In [ ]:
# === CELL 6 — sanity check ===
records = [json.loads(l) for l in open(OUTPUT_JSONL, encoding="utf-8") if l.strip()]
no_majority = sum(1 for r in records if r["majority_label"] is None)
tiers = [r["majority_label"]["risk_tier"] for r in records if r["majority_label"]]
audit = sum(1 for r in records if r["requires_human_audit"])
print(f"records={len(records)} no_valid_majority={no_majority} requires_human_audit={audit}")
print("tier distribution:", {t: tiers.count(t) for t in sorted(set(tiers))})

## 7. Hand off

Download `raw_primary_runs_medgemma27b.jsonl` from `/kaggle/working`, then
continue the pipeline unchanged on your own machine/repo:

```bash
python src/teacher_judging/judge_aux_labels.py \
  --input data/synthetic_aux/raw_primary_runs_medgemma27b.jsonl \
  --output data/synthetic_aux/judged_medgemma27b.jsonl \
  ...

python src/teacher_labeling/apply_quality_gates.py ...
python src/teacher_labeling/build_student_sft_jsonl.py ...
```